In [ ]:
!pip install haversine

In [ ]:
import os
import pandas as pd
import numpy as np
import datetime
from scipy import sparse
import matplotlib.pyplot as plt
from haversine import haversine, Unit
from collections import deque
import heapq

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/Math168/fraud_detection/"
print(os.listdir(path))

['fraudTrain.csv', 'fraudTest.csv', 'tx_birank_priors.csv', 'tx_birank_priors_train.csv', 'tx_birank_priors_test1m.csv']


In [ ]:
fraudTest = pd.read_csv(path + "fraudTest.csv", index_col=0)
fraudTrain = pd.read_csv(path + "fraudTrain.csv", index_col=0)
fraudTrain

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296670,2020-06-21 12:12:08,30263540414123,fraud_Reichel Inc,entertainment,15.56,Erik,Patterson,M,162 Jessica Row Apt. 072,Hatch,...,37.7175,-112.4777,258,Geoscientist,1961-11-24,440b587732da4dc1a6395aba5fb41669,1371816728,36.841266,-111.690765,0
1296671,2020-06-21 12:12:19,6011149206456997,fraud_Abernathy and Sons,food_dining,51.70,Jeffrey,White,M,8617 Holmes Terrace Suite 651,Tuscarora,...,39.2667,-77.5101,100,"Production assistant, television",1979-12-11,278000d2e0d2277d1de2f890067dcc0a,1371816739,38.906881,-78.246528,0
1296672,2020-06-21 12:12:32,3514865930894695,fraud_Stiedemann Ltd,food_dining,105.93,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,...,32.9396,-105.8189,899,Naval architect,1967-08-30,483f52fe67fabef353d552c1e662974c,1371816752,33.619513,-105.130529,0
1296673,2020-06-21 12:13:36,2720012583106919,"fraud_Reinger, Weissnat and Strosin",food_dining,74.90,Joseph,Murray,M,42933 Ryan Underpass,Manderson,...,43.3526,-102.5411,1126,Volunteer coordinator,1980-08-18,d667cdcbadaaed3da3f4020e83591c83,1371816816,42.788940,-103.241160,0


In [ ]:
fraudTrain['source'] = 'train'
fraudTest['source'] = 'test'
fraudTrain['orig_index'] = fraudTrain.index
fraudTest['orig_index'] = fraudTest.index

In [ ]:
fraudTrain['trans_date_trans_time'] = pd.to_datetime(fraudTrain['trans_date_trans_time'], errors='coerce')
fraudTest['trans_date_trans_time'] = pd.to_datetime(fraudTest['trans_date_trans_time'], errors='coerce')

In [ ]:
test_start = fraudTest['trans_date_trans_time'].min()
test_end = test_start + pd.DateOffset(months=1)   # first calendar month

fraudTest_1m = fraudTest[
    (fraudTest['trans_date_trans_time'] >= test_start) &
    (fraudTest['trans_date_trans_time'] < test_end)
].copy()

print("fraudTrain rows:", len(fraudTrain))
print("fraudTest total rows:", len(fraudTest))
print("fraudTest first-month rows:", len(fraudTest_1m))
print("Test window:", test_start, "to", test_end)

fraudTrain rows: 1296675
fraudTest total rows: 555719
fraudTest first-month rows: 87645
Test window: 2020-06-21 12:14:25 to 2020-07-21 12:14:25


In [ ]:
fraudAll = pd.concat([fraudTrain, fraudTest_1m], axis=0)

In [ ]:
fraudAll = fraudAll.reset_index(drop=True)
fraudAll['row_id_combined'] = fraudAll.index

In [ ]:
fraudAll

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,source,orig_index,row_id_combined
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,train,0,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,train,1,1
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,train,2,2
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,train,3,3
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,train,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1384315,2020-07-21 12:13:27,6011366578560244,"fraud_Huel, Hammes and Witting",grocery_pos,102.10,Adam,Stark,M,0912 Mark Fields Apt. 080,Mc Veytown,...,Nutritional therapist,1997-07-01,28dbfcfd6288b406e125154d26a8048b,1374408807,41.103110,-77.601875,0,test,87640,1384315
1384316,2020-07-21 12:13:30,2264937662466770,fraud_Labadie LLC,personal_care,7.95,Juan,Sherman,M,5939 Garcia Forges Suite 297,San Antonio,...,Land,1995-10-17,a057b7eecf1683d1109afd59d6478d6a,1374408810,29.993934,-98.163099,0,test,87641,1384316
1384317,2020-07-21 12:13:32,30082025922891,fraud_Ernser-Feest,home,123.11,Kathleen,Thompson,F,199 Patterson Fords Apt. 132,Naples,...,"Pilot, airline",1934-06-23,ba8f31627906ee914d3150b3557ebe03,1374408812,25.973459,-81.848113,0,test,87642,1384317
1384318,2020-07-21 12:13:36,30235438713303,"fraud_Kerluke, Considine and Macejkovic",misc_net,811.45,James,Baldwin,M,3603 Mitchell Court,Winfield,...,Exhibition designer,1980-03-24,014019287afe680e698414b3ca38dc09,1374408816,37.583384,-82.006009,0,test,87643,1384318


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 6371.0 * (2 * np.arcsin(np.sqrt(a)))

### EDA / Shared Plot

In [ ]:
# fraudTrain['trans_date_trans_time'] = pd.to_datetime(fraudTrain['trans_date_trans_time'], errors='coerce')

check = pd.DataFrame({
    'string_time': fraudAll['trans_date_trans_time'],
    'unix_time_converted': pd.to_datetime(fraudAll['unix_time'], unit='s', errors='coerce')
})

print(check.head())
print((check['string_time'] - check['unix_time_converted']).dropna().describe())

          string_time unix_time_converted
0 2019-01-01 00:00:18 2012-01-01 00:00:18
1 2019-01-01 00:00:44 2012-01-01 00:00:44
2 2019-01-01 00:00:51 2012-01-01 00:00:51
3 2019-01-01 00:01:16 2012-01-01 00:01:16
4 2019-01-01 00:03:06 2012-01-01 00:03:06
count                         1384320
mean     2556 days 07:58:30.686546464
std         0 days 11:18:17.677969606
min                2556 days 00:00:00
25%                2556 days 00:00:00
50%                2556 days 00:00:00
75%                2557 days 00:00:00
max                2557 days 00:00:00
dtype: object


In [ ]:
feat_df = pd.DataFrame({
    'trans_num': fraudAll['trans_num'].values,
    'cc_num': fraudAll['cc_num'].values,
    'merchant': fraudAll['merchant'].values,
    'category': fraudAll['category'].values,
    'state': fraudAll['state'].values,

    'unix_time': fraudAll['unix_time'].values,                         # event clock (for differences)
    'trans_date_trans_time': fraudAll['trans_date_trans_time'].values, # calendar clock (for hour/day)

    'is_fraud': fraudAll['is_fraud'].astype(int).values,
    'source': fraudAll['source'].values,
    'orig_index': fraudAll['orig_index'].values,
    'row_id_combined': fraudAll['row_id_combined'].values

})


### Card Behavior / Amount Anomaly

In [ ]:
raw = fraudAll.copy().reset_index(drop=False).rename(columns={'index': 'row_id'})

In [ ]:
# ordered first by card number, then time
raw_card = raw.sort_values(['cc_num', 'unix_time'])
## recenvy, dist_prev_merchant_km, implied_speed_kmh, amt_to_card_median

# ordered first by time, then card number
raw_global = raw.sort_values(['unix_time', 'cc_num'])

In [ ]:
feat_df['amt'] = fraudAll['amt'].values
feat_df['log_amt'] = np.log1p(fraudAll['amt'].clip(lower=0).values)

In [ ]:
"""left = raw_global[['unix_time', 'cc_num']].reset_index(drop=True)
right = fraudTrain[['unix_time', 'cc_num']].sort_values(['unix_time', 'cc_num']).reset_index(drop=True)
print(left.equals(right))"""

"left = raw_global[['unix_time', 'cc_num']].reset_index(drop=True)\nright = fraudTrain[['unix_time', 'cc_num']].sort_values(['unix_time', 'cc_num']).reset_index(drop=True)\nprint(left.equals(right))"

In [ ]:
## ratio to historical (past-only) median purchasing price
raw_card['card_past_median_amt'] = (
    raw_card.groupby('cc_num')['amt']
            .expanding()
            .median() # cumulative median within each card in card_time_order
            .reset_index(level=0, drop=True)
            .groupby(raw_card['cc_num'])
            .shift(1)
)
raw_card['amt_to_card_median'] = raw_card['amt'] / raw_card['card_past_median_amt']
raw_card['amt_to_card_median_missing'] = raw_card['card_past_median_amt'].isna().astype(int)

In [ ]:
feat_df.loc[raw_card['row_id'], 'card_past_median_amt'] = raw_card['card_past_median_amt'].values
feat_df.loc[raw_card['row_id'], 'amt_to_card_median'] = raw_card['amt_to_card_median'].values
feat_df.loc[raw_card['row_id'], 'amt_to_card_median_missing'] = raw_card['amt_to_card_median_missing'].values

In [ ]:
# indicators for short term frequent purchases
## card transaction count in 1 hour
## unique merchants in 1 hour
raw_card['card_txn_count_1h'] = 0
raw_card['card_unique_merchants_1h'] = 0
WINDOW_SEC = 3600
for cc, g in raw_card.groupby('cc_num', sort=False): # credit card number, group (rows) for that credit card
  idx = g.index.to_list()
  times = raw_card.loc[idx, 'unix_time'].to_numpy()
  merchants = raw_card.loc[idx, 'merchant'].to_numpy()

  q = deque()
  merchant_freq = {}

  for pos, row in enumerate(idx): # position, value
    t = times[pos]
    while q and (t - times[q[0]] > WINDOW_SEC): # current time older than 1h
      old_pos = q.popleft()
      old_m = merchants[old_pos]
      merchant_freq[old_m] -= 1
      if merchant_freq[old_m] == 0:
        merchant_freq.pop(old_m)

    raw_card.at[row, 'card_txn_count_1h'] = len(q)
    raw_card.at[row, 'card_unique_merchants_1h'] = len(merchant_freq)

    q.append(pos)
    merchant_freq[merchants[pos]] = merchant_freq.get(merchants[pos], 0) + 1

feat_df.loc[raw_card['row_id'], 'card_txn_count_1h'] = raw_card['card_txn_count_1h'].values
feat_df.loc[raw_card['row_id'], 'card_unique_merchants_1h'] = raw_card['card_unique_merchants_1h'].values

### Geographic / Regional Context

(fraudulent transactions in given state + alpha * global fraudulent percentage) / (total transactions in given state in past + alpha)

In [ ]:
raw_global['hour_of_day'] = raw_global['trans_date_trans_time'].dt.hour.values

In [ ]:
alpha = 50
global_base_rate = raw_global['is_fraud'].mean()

# Initialize columns
raw_global['state_fraud_rate_past_smoothed'] = np.nan
raw_global['category_fraud_rate_past_smoothed'] = np.nan
raw_global['hour_fraud_rate_past_smoothed'] = np.nan

# Running counts
state_counts = {}
state_frauds = {}

cat_counts = {}
cat_frauds = {}

hour_counts = {}
hour_frauds = {}

for ridx in raw_global.index:
    state = raw_global.at[ridx, 'state']
    cat = raw_global.at[ridx, 'category']
    hour = raw_global.at[ridx, 'hour_of_day']
    y = int(raw_global.at[ridx, 'is_fraud'])

    # Past-only smoothed rates for current row
    sN = state_counts.get(state, 0)
    sF = state_frauds.get(state, 0)
    raw_global.at[ridx, 'state_fraud_rate_past_smoothed'] = (sF + alpha * global_base_rate) / (sN + alpha)

    cN = cat_counts.get(cat, 0)
    cF = cat_frauds.get(cat, 0)
    raw_global.at[ridx, 'category_fraud_rate_past_smoothed'] = (cF + alpha * global_base_rate) / (cN + alpha)

    hN = hour_counts.get(hour, 0)
    hF = hour_frauds.get(hour, 0)
    raw_global.at[ridx, 'hour_fraud_rate_past_smoothed'] = (hF + alpha * global_base_rate) / (hN + alpha)

    # update counts after
    state_counts[state] = sN + 1
    state_frauds[state] = sF + y

    cat_counts[cat] = cN + 1
    cat_frauds[cat] = cF + y

    hour_counts[hour] = hN + 1
    hour_frauds[hour] = hF + y

# Map back using row_id
feat_df.loc[raw_global['row_id'], 'state_fraud_rate_past_smoothed'] = raw_global['state_fraud_rate_past_smoothed'].values
feat_df.loc[raw_global['row_id'], 'category_fraud_rate_past_smoothed'] = raw_global['category_fraud_rate_past_smoothed'].values
feat_df.loc[raw_global['row_id'], 'hour_fraud_rate_past_smoothed'] = raw_global['hour_fraud_rate_past_smoothed'].values

### Merchant / Category Context

In [ ]:
raw_global['merchant_txn_count_past'] = 0
raw_global['merchant_unique_cards_past'] = 0

merchant_txn_count = {}    # count of transaction
merchant_cards_seen = {}   # set of cards seen so far

for ridx in raw_global.index:
    m = raw_global.at[ridx, 'merchant']
    cc = raw_global.at[ridx, 'cc_num']

    past_txn_count = merchant_txn_count.get(m, 0)
    past_card_set = merchant_cards_seen.get(m, set())

    raw_global.at[ridx, 'merchant_txn_count_past'] = past_txn_count
    raw_global.at[ridx, 'merchant_unique_cards_past'] = len(past_card_set)

    merchant_txn_count[m] = past_txn_count + 1

    if m not in merchant_cards_seen:
        merchant_cards_seen[m] = set()
    merchant_cards_seen[m].add(cc)

# Map back to feat_df using row_id
feat_df.loc[raw_global['row_id'], 'merchant_txn_count_past'] = raw_global['merchant_txn_count_past'].values
feat_df.loc[raw_global['row_id'], 'merchant_unique_cards_past'] = raw_global['merchant_unique_cards_past'].values

In [ ]:
## amt_to_category_median
class RunningMedian:
    def __init__(self):
        self.low = [] # max-heap for lower half
        self.high = [] # min-heap for upper half

    def add(self, x: float):
        if not self.low or x <= -self.low[0]:
            heapq.heappush(self.low, -x)
        else:
            heapq.heappush(self.high, x)

        if len(self.low) < len(self.high):
            heapq.heappush(self.low, -heapq.heappop(self.high))
        elif len(self.low) > len(self.high) + 1:
            heapq.heappush(self.high, -heapq.heappop(self.low))

    def median(self):
        n_low = len(self.low)
        n_high = len(self.high)
        if n_low == 0 and n_high == 0:
            return np.nan
        if n_low > n_high:
            return float(-self.low[0])
        return float((-self.low[0] + self.high[0]) / 2.0)

    def __len__(self):
        return len(self.low) + len(self.high)

In [ ]:
n = len(raw_global)
category_past_median_amt = np.full(n, np.nan, dtype=float)
amt_to_category_median = np.full(n, np.nan, dtype=float)

category_medians = {}

In [ ]:
for i, ridx in enumerate(raw_global.index):
    cat = raw_global.at[ridx, 'category']
    amt = float(raw_global.at[ridx, 'amt'])

    rm = category_medians.get(cat)
    if rm is None:
        rm = RunningMedian()
        category_medians[cat] = rm

    if len(rm) > 0:
        med = rm.median()
        category_past_median_amt[i] = med
        amt_to_category_median[i] = amt / med if med > 0 else np.nan

    rm.add(amt)

In [ ]:
raw_global['category_past_median_amt'] = category_past_median_amt
raw_global['amt_to_category_median'] = amt_to_category_median

feat_df.loc[raw_global['row_id'], 'category_past_median_amt'] = raw_global['category_past_median_amt'].to_numpy()
feat_df.loc[raw_global['row_id'], 'amt_to_category_median'] = raw_global['amt_to_category_median'].to_numpy()

In [ ]:
"""## amt_to_category_median
raw_global['category_past_median_amt'] = np.nan
raw_global['amt_to_category_median'] = np.nan

category_amt_history = {}

for ridx in raw_global.index:
    cat = raw_global.at[ridx, 'category']
    amt = float(raw_global.at[ridx, 'amt'])

    past_amts = category_amt_history.get(cat, [])

    if len(past_amts) > 0:
        med = float(np.median(np.asarray(past_amts, dtype=float)))
        raw_global.at[ridx, 'category_past_median_amt'] = med
        raw_global.at[ridx, 'amt_to_category_median'] = amt / med if med > 0 else np.nan
    else:
        raw_global.at[ridx, 'category_past_median_amt'] = np.nan
        raw_global.at[ridx, 'amt_to_category_median'] = np.nan

    if cat not in category_amt_history:
        category_amt_history[cat] = []
    category_amt_history[cat].append(amt)

feat_df.loc[raw_global['row_id'], 'category_past_median_amt'] = raw_global['category_past_median_amt'].values
feat_df.loc[raw_global['row_id'], 'amt_to_category_median'] = raw_global['amt_to_category_median'].values"""

"## amt_to_category_median\nraw_global['category_past_median_amt'] = np.nan\nraw_global['amt_to_category_median'] = np.nan\n\ncategory_amt_history = {}\n\nfor ridx in raw_global.index:\n    cat = raw_global.at[ridx, 'category']\n    amt = float(raw_global.at[ridx, 'amt'])\n\n    past_amts = category_amt_history.get(cat, [])\n\n    if len(past_amts) > 0:\n        med = float(np.median(np.asarray(past_amts, dtype=float)))\n        raw_global.at[ridx, 'category_past_median_amt'] = med\n        raw_global.at[ridx, 'amt_to_category_median'] = amt / med if med > 0 else np.nan\n    else:\n        raw_global.at[ridx, 'category_past_median_amt'] = np.nan\n        raw_global.at[ridx, 'amt_to_category_median'] = np.nan\n\n    if cat not in category_amt_history:\n        category_amt_history[cat] = []\n    category_amt_history[cat].append(amt)\n\nfeat_df.loc[raw_global['row_id'], 'category_past_median_amt'] = raw_global['category_past_median_amt'].values\nfeat_df.loc[raw_global['row_id'], 'amt_to_

### Time & Temporal Patterns

In [ ]:
feat_df['hour_of_day'] = fraudAll['trans_date_trans_time'].dt.hour.values
feat_df['day_of_week'] = fraudAll['trans_date_trans_time'].dt.dayofweek.values
feat_df['is_weekend'] = fraudAll['trans_date_trans_time'].dt.dayofweek.isin([5, 6]).astype(int).values

In [ ]:
feat_df['dist_home_merchant_km'] = haversine_km(
    fraudAll['lat'].values, fraudAll['long'].values,
    fraudAll['merch_lat'].values, fraudAll['merch_long'].values
)

In [ ]:
## previous transaction time and previous merchant location within the same card
raw_card['prev_unix'] = raw_card.groupby('cc_num')['unix_time'].shift(1)
raw_card['prev_merch_lat'] = raw_card.groupby('cc_num')['merch_lat'].shift(1)
raw_card['prev_merch_long'] = raw_card.groupby('cc_num')['merch_long'].shift(1)

In [ ]:
## recency (seconds)
raw_card['recency_sec'] = raw_card['unix_time'] - raw_card['prev_unix']

In [ ]:
## distance from previous merchant
raw_card['dist_prev_merchant_km'] = np.nan
mask_prev = raw_card['prev_merch_lat'].notna() & raw_card['prev_merch_long'].notna()

raw_card.loc[mask_prev, 'dist_prev_merchant_km'] = haversine_km(
    raw_card.loc[mask_prev, 'prev_merch_lat'].values,
    raw_card.loc[mask_prev, 'prev_merch_long'].values,
    raw_card.loc[mask_prev, 'merch_lat'].values,     # <- use raw_card, not fraudTrain
    raw_card.loc[mask_prev, 'merch_long'].values
)

In [ ]:
## previous transaction implied speed for cardholder
raw_card['implied_speed_kmh'] = np.nan
valid_speed = (
    raw_card['recency_sec'].notna()
    & (raw_card['recency_sec'] > 0)
    & raw_card['dist_prev_merchant_km'].notna()
)

raw_card.loc[valid_speed, 'implied_speed_kmh'] = (
    raw_card.loc[valid_speed, 'dist_prev_merchant_km'] /
    (raw_card.loc[valid_speed, 'recency_sec'] / 3600.0)
)

In [ ]:
# Assign sequence features back to feat_df using original row identity
feat_df.loc[raw_card['row_id'], 'recency_sec'] = raw_card['recency_sec'].values
feat_df.loc[raw_card['row_id'], 'dist_prev_merchant_km'] = raw_card['dist_prev_merchant_km'].values
feat_df.loc[raw_card['row_id'], 'implied_speed_kmh'] = raw_card['implied_speed_kmh'].values

In [ ]:
print(feat_df[['hour_of_day','day_of_week','log_amt','dist_home_merchant_km',
               'recency_sec','dist_prev_merchant_km','implied_speed_kmh','is_fraud']].head())

print("\nMissingness:")
print(feat_df[['recency_sec','dist_prev_merchant_km','implied_speed_kmh']].isna().mean())

print("\nDescribe:")
print(feat_df[['amt','log_amt','dist_home_merchant_km','recency_sec',
               'dist_prev_merchant_km','implied_speed_kmh']].describe(percentiles=[.5,.9,.95,.99]).T)

   hour_of_day  day_of_week   log_amt  dist_home_merchant_km  recency_sec  \
0            0            1  1.786747              78.597568          NaN   
1            0            1  4.684259              30.212176          NaN   
2            0            1  5.398660             108.206083          NaN   
3            0            1  3.828641              95.673231          NaN   
4            0            1  3.760269              77.556744          NaN   

   dist_prev_merchant_km  implied_speed_kmh  is_fraud  
0                    NaN                NaN         0  
1                    NaN                NaN         0  
2                    NaN                NaN         0  
3                    NaN                NaN         0  
4                    NaN                NaN         0  

Missingness:
recency_sec              0.000712
dist_prev_merchant_km    0.000712
implied_speed_kmh        0.000730
dtype: float64

Describe:
                           count          mean           st

In [ ]:
feat_df.columns

Index(['trans_num', 'cc_num', 'merchant', 'category', 'state', 'unix_time',
       'trans_date_trans_time', 'is_fraud', 'source', 'orig_index',
       'row_id_combined', 'amt', 'log_amt', 'card_past_median_amt',
       'amt_to_card_median', 'amt_to_card_median_missing', 'card_txn_count_1h',
       'card_unique_merchants_1h', 'state_fraud_rate_past_smoothed',
       'category_fraud_rate_past_smoothed', 'hour_fraud_rate_past_smoothed',
       'merchant_txn_count_past', 'merchant_unique_cards_past',
       'category_past_median_amt', 'amt_to_category_median', 'hour_of_day',
       'day_of_week', 'is_weekend', 'dist_home_merchant_km', 'recency_sec',
       'dist_prev_merchant_km', 'implied_speed_kmh'],
      dtype='object')

### Split back after features are done

In [ ]:
feat_train = feat_df[feat_df['source'] == 'train'].copy()
feat_test_1m = feat_df[feat_df['source'] == 'test'].copy()

# Optional: restore original file order
feat_train = feat_train.sort_values('orig_index')
feat_test_1m = feat_test_1m.sort_values('orig_index')

print("feat_train shape:", feat_train.shape)
print("feat_test_1m shape:", feat_test_1m.shape)

feat_train shape: (1296675, 32)
feat_test_1m shape: (87645, 32)


In [ ]:
train_csv = path + "feat_train.csv"
test_csv  = path + "feat_test1m.csv"

feat_train.to_csv(train_csv)
feat_test_1m.to_csv(test_csv)

print("Wrote:", train_csv)
print("Wrote:", test_csv)

Wrote: /content/drive/MyDrive/Math168/fraud_detection/feat_train.csv
Wrote: /content/drive/MyDrive/Math168/fraud_detection/feat_test1m.csv


In [ ]:
from google.colab import files
files.download(train_csv)
files.download(test_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>